<a href="https://colab.research.google.com/github/Elwing-Chou/tibame_20260714/blob/main/tibame20260728.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pygame as pg
import random


#pygame初始化
pg.init()

# --- 字型 ---
def get_font(size):
    for f in ['microsoftjhenghei', 'simhei', 'stheitirelight']:
        if f in pg.font.get_fonts():
            return pg.font.SysFont(f, size)
    return pg.font.SysFont(None, size)


# GAME常數
FONT_UI = get_font(48)
col, row = 8, 8
inter = 80
width, height = (col + 2) * inter, (row + 2) * inter

# GAME LOGIC
board_nums = [[None] * col for i in range(row)]
NOT_PAIR, PAIR = False, True
board_pair = [[NOT_PAIR] * col for i in range(row)]
# 整個遊戲的最重要的地方, 是我新的位置要跟上一個位置做對比(是否是個pair)
i_prev, j_prev = None, None

# 我會先把所有 (i, j) 放成一個list
total_pos = []
for i in range(row):
    for j in range(col):
        total_pos.append((i, j))
random.shuffle(total_pos)
# print(total_pos)

# list.pop()
# 一次pop出兩個, 把這兩個在board_nums設成同一個數字
# 因為數字不能重複, 所以你要拿一個list/set紀錄你曾經random.randint(1, 100)的數字
already = set()
while len(total_pos) > 0:
    i1, j1 = total_pos.pop()
    i2, j2 = total_pos.pop()
    # 不斷產生新數字, 直到這個數字不在already裡面
    while True:
        n = random.randint(1, 100)
        if not n in already:
            already.add(n)
            board_nums[i1][j1] = n
            board_nums[i2][j2] = n
            break
print(board_nums)


# GAME UI
# 產生視窗
screen = pg.display.set_mode((width, height))
# 設定遊戲標題
pg.display.set_caption("翻牌")

def draw():
    # 準備第一個圖層
    bg = pg.Surface(screen.get_size())
    # 把畫布填滿某個顏色
    bg.fill((199, 167, 82))

    # x_ul, y_ul: 測試 -> 換成真正位置(i_prev/j_prev)
    if i_prev == None or j_prev == None:
        pass
    else:
        x_ul, y_ul = (j_prev + 1) * inter, (i_prev + 1) * inter
        pg.draw.rect(bg, (255, 255, 255), (x_ul, y_ul, inter, inter), 0)
        # 測試一下秀出來的數字
        t = FONT_UI.render(str(board_nums[i_prev][j_prev]), 1, (0, 0, 0))
        # 背景(bg)上面疊上t get_rect(中心座標) -> 左上角座標
        bg.blit(t, t.get_rect(center=(x_ul+inter/2, y_ul+inter/2)))

    # (new) 把pair的都畫出來: 畫圖程式碼跟上面一樣
    for i in range(row):
        for j in range(col):
            if board_pair[i][j] == PAIR:
                x_ul, y_ul = (j + 1) * inter, (i + 1) * inter
                pg.draw.rect(bg, (255, 255, 255), (x_ul, y_ul, inter, inter), 0)
                # 測試一下秀出來的數字
                t = FONT_UI.render(str(board_nums[i][j]), 1, (0, 0, 0))
                # 背景(bg)上面疊上t get_rect(中心座標) -> 左上角座標
                bg.blit(t, t.get_rect(center=(x_ul + inter / 2, y_ul + inter / 2)))

    # 畫橫線
    # pygame.draw.line(畫布, 顏色, (x坐標1, y坐標1), (x坐標2, y坐標2), 線寬)
    for i in range(row+1):
        pg.draw.line(bg,
                     (0, 0, 0),
                     (inter, inter*i+inter),
                     (width-inter, inter*i+inter),
                     1)

    # 畫直線
    for i in range(col+1):
        pg.draw.line(bg,
                     (0, 0, 0),
                     (inter*i+inter, inter),
                     (inter*i+inter, height-inter),
                     1)

    # 你要把圖層放到上一層
    screen.blit(bg, (0, 0))
    # 對畫面進行更新(才會真的秀出來)
    pg.display.update()

# (我最後加)
def check_win():
    # 只要裡面還有一個是未配對, 就沒有贏
    for i in range(row):
        for j in range(col):
            if board_pair[i][j] == NOT_PAIR:
                return False
    # 所有位置都是pair
    return True


draw()
# 建立一個永不結束的迴圈(遊戲才不會結束)
running = True
while running:
    # 收取你的遊戲任何事件(滑鼠點擊/鍵盤按鈕...)
    for event in pg.event.get():
        # 偵測滑鼠點擊以後放掉的動作
        if event.type == pg.MOUSEBUTTONUP:
            x, y = pg.mouse.get_pos()
            # 把x, y座標換成我們邏輯裡面的雙層list的位置 i(跟y), j(跟x)
            i_mouse, j_mouse = y // inter - 1, x // inter - 1
            # 什麼時候點的是合法的位置
            # 1. 點跟上一個位置同樣地方: 不做事
            # 2. 點是已經被配對過的: x
            if board_pair[i_mouse][j_mouse] == NOT_PAIR:
                # 跟上一個位置做比對
                # 1. 一樣數字: PAIR
                # 2. 不一樣: 無事發生
                # 還沒有點過任何一個地方
                if i_prev == None or j_prev == None:
                    print("還沒有任何被點過的人")
                elif i_prev == i_mouse and j_prev == j_mouse:
                    print("沒換位置")
                else:
                    if board_nums[i_mouse][j_mouse] == board_nums[i_prev][j_prev]:
                        # 既然一樣, 這兩個地方的pair都設定起來
                        board_pair[i_mouse][j_mouse] = PAIR
                        board_pair[i_prev][j_prev] = PAIR
                # 把上一個位置更新成新的位置
                i_prev, j_prev = i_mouse, j_mouse
            # 根據邏輯進行重繪
            draw()
            # check 有沒有贏
            if check_win() == True:
                # 結束遊戲
                print("結束遊戲")
                running = False
        # 如果收到的事件是按x
        if event.type == pg.QUIT:
            # 迴圈就會變成while False
            running = False

pg.quit()

In [ ]:
# reset版本(參考即可)
import pygame as pg
import random


#pygame初始化
pg.init()

# --- 字型 ---
def get_font(size):
    for f in ['microsoftjhenghei', 'simhei', 'stheitirelight']:
        if f in pg.font.get_fonts():
            return pg.font.SysFont(f, size)
    return pg.font.SysFont(None, size)


# GAME常數
FONT_UI = get_font(48)
col, row = 8, 8
inter = 80
width, height = (col + 2) * inter, (row + 2) * inter


def reset():
    board_nums = [[None] * col for i in range(row)]
    board_pair = [[NOT_PAIR] * col for i in range(row)]
    # 整個遊戲的最重要的地方, 是我新的位置要跟上一個位置做對比(是否是個pair)
    # 我會先把所有 (i, j) 放成一個list
    total_pos = []
    for i in range(row):
        for j in range(col):
            total_pos.append((i, j))
    random.shuffle(total_pos)
    already = set()
    while len(total_pos) > 0:
        i1, j1 = total_pos.pop()
        i2, j2 = total_pos.pop()
        # 不斷產生新數字, 直到這個數字不在already裡面
        while True:
            n = random.randint(1, 100)
            if not n in already:
                already.add(n)
                board_nums[i1][j1] = n
                board_nums[i2][j2] = n
                break
    return (board_nums, board_pair, None, None)

# GAME LOGIC
NOT_PAIR, PAIR = False, True
# 整個遊戲的最重要的地方, 是我新的位置要跟上一個位置做對比(是否是個pair)
board_nums, board_pair, i_prev, j_prev = reset()

# 我會先把所有 (i, j) 放成一個list
total_pos = []
for i in range(row):
    for j in range(col):
        total_pos.append((i, j))
random.shuffle(total_pos)


# GAME UI
# 產生視窗
screen = pg.display.set_mode((width, height))
# 設定遊戲標題
pg.display.set_caption("翻牌")

def draw():
    # 準備第一個圖層
    bg = pg.Surface(screen.get_size())
    # 把畫布填滿某個顏色
    bg.fill((199, 167, 82))

    # x_ul, y_ul: 測試 -> 換成真正位置(i_prev/j_prev)
    if i_prev == None or j_prev == None:
        pass
    else:
        x_ul, y_ul = (j_prev + 1) * inter, (i_prev + 1) * inter
        pg.draw.rect(bg, (255, 255, 255), (x_ul, y_ul, inter, inter), 0)
        # 測試一下秀出來的數字
        t = FONT_UI.render(str(board_nums[i_prev][j_prev]), 1, (0, 0, 0))
        # 背景(bg)上面疊上t get_rect(中心座標) -> 左上角座標
        bg.blit(t, t.get_rect(center=(x_ul+inter/2, y_ul+inter/2)))

    # (new) 把pair的都畫出來: 畫圖程式碼跟上面一樣
    for i in range(row):
        for j in range(col):
            if board_pair[i][j] == PAIR:
                x_ul, y_ul = (j + 1) * inter, (i + 1) * inter
                pg.draw.rect(bg, (255, 255, 255), (x_ul, y_ul, inter, inter), 0)
                # 測試一下秀出來的數字
                t = FONT_UI.render(str(board_nums[i][j]), 1, (0, 0, 0))
                # 背景(bg)上面疊上t get_rect(中心座標) -> 左上角座標
                bg.blit(t, t.get_rect(center=(x_ul + inter / 2, y_ul + inter / 2)))

    # 畫橫線
    # pygame.draw.line(畫布, 顏色, (x坐標1, y坐標1), (x坐標2, y坐標2), 線寬)
    for i in range(row+1):
        pg.draw.line(bg,
                     (0, 0, 0),
                     (inter, inter*i+inter),
                     (width-inter, inter*i+inter),
                     1)

    # 畫直線
    for i in range(col+1):
        pg.draw.line(bg,
                     (0, 0, 0),
                     (inter*i+inter, inter),
                     (inter*i+inter, height-inter),
                     1)

    # 你要把圖層放到上一層
    screen.blit(bg, (0, 0))
    # 對畫面進行更新(才會真的秀出來)
    pg.display.update()

# (我最後加)
def check_win():
    # 只要裡面還有一個是未配對, 就沒有贏
    for i in range(row):
        for j in range(col):
            if board_pair[i][j] == NOT_PAIR:
                return False
    # 所有位置都是pair
    return True


draw()
# 建立一個永不結束的迴圈(遊戲才不會結束)
running = True
while running:
    # 收取你的遊戲任何事件(滑鼠點擊/鍵盤按鈕...)
    for event in pg.event.get():
        # 偵測滑鼠點擊以後放掉的動作
        if event.type == pg.MOUSEBUTTONUP:
            x, y = pg.mouse.get_pos()
            # 把x, y座標換成我們邏輯裡面的雙層list的位置 i(跟y), j(跟x)
            i_mouse, j_mouse = y // inter - 1, x // inter - 1
            # 什麼時候點的是合法的位置
            # 1. 點跟上一個位置同樣地方: 不做事
            # 2. 點是已經被配對過的: x
            if board_pair[i_mouse][j_mouse] == NOT_PAIR:
                # 跟上一個位置做比對
                # 1. 一樣數字: PAIR
                # 2. 不一樣: 無事發生
                # 還沒有點過任何一個地方
                if i_prev == None or j_prev == None:
                    print("還沒有任何被點過的人")
                elif i_prev == i_mouse and j_prev == j_mouse:
                    print("沒換位置")
                else:
                    if board_nums[i_mouse][j_mouse] == board_nums[i_prev][j_prev]:
                        # 既然一樣, 這兩個地方的pair都設定起來
                        board_pair[i_mouse][j_mouse] = PAIR
                        board_pair[i_prev][j_prev] = PAIR
                # 把上一個位置更新成新的位置
                i_prev, j_prev = i_mouse, j_mouse
            # 根據邏輯進行重繪
            draw()
            # check 有沒有贏
            if check_win() == True:
                # 結束遊戲
                board_nums, board_pair, i_prev, j_prev = reset()
                draw()
        # 如果收到的事件是按x
        if event.type == pg.QUIT:
            # 迴圈就會變成while False
            running = False

pg.quit()